# Final Project

Movie review sentiment analysis.

In [6]:
from pathlib import Path
import jsonlines
import json
import itertools
import pprint
import re
import textwrap


import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import StackingClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline, FeatureUnion


In [7]:
def load_data(path):
    df = pd.read_csv(path).rename(columns={k: k.lower() for k in ['ID', 'TEXT', 'LABEL']})
    return df

df = load_data('data/train.csv')

# Just document this so you don't get it backwards latter: LABELS[int] gets sentiment.
LABELS = ['non-movie', 'postive', 'negative']
df.head()

,id,text,label
0,327995652116863138,Carolyn Baker defines the niche of helping peo...,0
1,11848336500074516230,Just getting released from a six month drug re...,1
2,8453485550425672763,I was greatly dissappointed when I popped this...,0
3,13088897910749354342,This is a film that has garnered any interest ...,2
4,4199320520317837800,This is one of the more adorable episodes of t...,1


In [9]:
# there are a few null valued samples.
if df.isna().sum().sum():
    df = df.dropna()

In [ ]:
def describe_data(df: pd.DataFrame):
    text_to_vect = CountVectorizer(ngram_range=(1, 1), min_df=1, stop_words='english')
    text_to_vect.fit(df.text)
    label_frequencies = df.label.value_counts(normalize=True).rename({i: v for i, v in enumerate(LABELS)})
    n_words = len(text_to_vect.vocabulary_)
    approximate_word_count = df.text.str.count('[\s.,;:_]')
    quantiles = [.025, .1, .5, .9, .975]
    quantile_values = approximate_word_count.quantile(quantiles)
    quantiles_summary = {q: v for q, v in zip(quantiles, quantile_values)}
    summary = textwrap.dedent(f"""
    Number of training samples: {len(df)}.
    The vocabulary size (not counting english stop words): {n_words}.
    The approximate words per text, as quantiles: {quantiles_summary}

    """) + f"The label frequencies are {label_frequencies}"
    print(summary)

describe_data(df)


Number of training samples: 70310.
The vocabulary size (not counting english stop words): 273952.
The approximate words per text, as quantiles: {0.025: 2.0, 0.1: 16.0, 0.5: 117.0, 0.9: 387.0, 0.975: 701.0}

The label frequencies are label
non-movie    0.459138
postive      0.272209
negative     0.268653
Name: proportion, dtype: float64


In [13]:
train, dev = train_test_split(df, random_state=879345)

## Modeling

In [14]:
class ExpressionTransformer(BaseEstimator, TransformerMixin):
    """Perform text feature extraction by labeling inputs that match expressions.

    The tranform method returns an n_samples x n_expressions array.
    """
    def __init__(self, expressions: list[str]):
        # This would be nice, but it runs into compilation errors with stacking models. );
        # self.expressions = [
        #     exp if isinstance(exp, re.Pattern)
        #     else re.compile(exp, re.IGNORECASE)
        #     for exp in expressions
        # ]
        self.expressions = expressions

    def fit(self, X, y=None):
        """Fit the model. This is a no-op and just returns self."""
        return self

    def transform(self, X):
        """Return a numpy array which encodes whether X matches each expression."""
        # NOTE: not tested much. May break given inputs that are multidimensions, etc.
        is_pandas = isinstance(X, pd.Series)
        if not is_pandas:
            X = pd.Series(X)
        transformed = []
        for expression in self.expressions:
            transformed.append(X.str.contains(expression).values)
        return np.stack(transformed).T


In [15]:
def make_model_pipeline(config: dict) -> BaseEstimator:
    """Build model from config.

    The configuration should specify a supported model type (see example below) as the 'model_type' entry.
    The rest of the entries are kwargs-like dictionaries passed to model components.

    Example Configurations
    ----------------------

        logistic_config = {
            'model_type': 'logistic_with_tfidf',
            'vectorizer': {'ngram_range': (1, 2), 'min_df': 10, 'stop_words': 'english'},
            'classifier': {'max_iter': 200},
            'expressions': {'expressions': [r'game|player', r'book|author|reader', r'song', r'dvd']}
        }
        multinomial_config = {
            'model_type': 'multinomial_count_vect',
            'vectorizer': {'ngram_range': (1, 2), 'min_df': 7, 'stop_words': 'english'},
            'classifier': {},
        }
        multinomial__with_features_config = {
            'model_type': 'multinomial_count_vect_with_features',
            'vectorizer': {'ngram_range': (1, 2), 'min_df': 7, 'stop_words': 'english'},
            'classifier': {},
            'expressions': {'expressions': [r'game|player', r'book|author|reader', r'song', r'dvd']}
        }
        model_config = {
            'model_type': 'voting',
            'logistic': logistic_config,
            'multinomial': multinomial_config
        }
    """
    config_type = config['model_type']

    if config_type == 'multinomial_count_vect':
        vectorizer_config = config['vectorizer']
        model_config = config['classifier']
        vectorizer = CountVectorizer(**vectorizer_config)
        classifier = MultinomialNB(**model_config)
        return Pipeline([('preprocessor', vectorizer), ('classifier', classifier)])
    elif config_type == 'multinomial_count_vect_with_features':
        vectorizer_config = config['vectorizer']
        model_config = config['classifier']
        vectorizer = CountVectorizer(**vectorizer_config)
        classifier = MultinomialNB(**model_config)
        expressions = ExpressionTransformer(**config['expressions'])
        pipeline = Pipeline([
            ('features', FeatureUnion([
                ('text', vectorizer),
                ('expressions', expressions),
            ])),
            ('clf', classifier)
        ])
        return pipeline
    elif config_type == 'logistic_with_tfidf':
        vectorizer_config = config['vectorizer']
        model_config = config['classifier']
        vectorizer = TfidfVectorizer(**vectorizer_config)
        classifier = LogisticRegression(**model_config)
        expressions = ExpressionTransformer(**config['expressions'])
        pipeline = Pipeline([
            ('features', FeatureUnion([
                ('text', vectorizer),
                ('expressions', expressions),
            ])),
            ('clf', classifier)
        ])
        return pipeline
    elif config_type == 'voting':
        multinomial = make_model_pipeline(config['multinomial'])
        logistic = make_model_pipeline(config['logistic'])
        return StackingClassifier(
            [
                ('multinomial', multinomial),
                ('logistic', logistic)],
            final_estimator=LogisticRegression()
        )
    raise ValueError(f"Unsupported model type {config['model_type']}")

def score_predictions(y_true, y_hat) -> dict:
    """Return a diction of f1-score and a confusion matrix."""
    return {
        'macro_f1': f1_score(y_true, y_hat, average='macro'),
        'confusion': confusion_matrix(y_true, y_hat).tolist()
    }


In [16]:
def experiment(model_config, train, test, experiments_dir='data/experiments'):
    """Run an experiment based on the model configuration.

    Returns the fitted model.
    """
    existing = get_experiment_results(model_config, experiments_dir)
    if existing:
        print(f"Experiment has already been run. Try something new.")
        print({k: v for k, v in existing.items() if k != 'config'})
        return
    model = make_model_pipeline(model_config)

    # fit model
    X = train.text
    y = train.label
    model.fit(X, y)

    # score model
    y_test = test.label
    X_test = test.text
    y_hat = model.predict(X_test)
    scores = score_predictions(y_test, y_hat)
    result = {'model_type': model_config['model_type'], 'config': model_config} | scores

    # save the results in to a JSON lines file.
    experiments_dir = Path(experiments_dir)
    if not experiments_dir.exists():
        experiments_dir.mkdir(parents=True)
    file_name = experiments_dir / 'experiments.jsonl'
    if not file_name.exists():
        file_name.touch()
    with jsonlines.open(file_name, 'a') as f:
        f.write(result)

    print(f"Experiment finished.\n{model_config}")
    print(f"{scores}")
    return model

def get_experiment_results(model_config, experiment_dir) -> dict:
    """Get the results of an experiment with model_config.

    Returns an empty dictionary if no experiment has been run.
    """
    experiment_dir = Path(experiment_dir)
    with jsonlines.open(experiment_dir / 'experiments.jsonl') as f:
        for experiment in f:
            experiment_config = experiment['config']

            if experiment_config.get('vectorizer'):
                ngram_range = experiment_config.get('vectorizer').get('ngram_range')
                if ngram_range:
                    experiment_config['vectorizer']['ngram_range'] = tuple(ngram_range)
            if experiment['config'] == model_config:
                return experiment
    return {}

def load_best_config(experiment_dir) -> dict:
    """Return the configuartion of the best experiment run."""
    experiment_dir = Path(experiment_dir)
    best = -np.inf
    best_experiment = None
    with jsonlines.open(experiment_dir / 'experiments.jsonl') as f:
        for experiment in f:
            f1 = experiment['macro_f1']
            if f1 > best:
                best = f1
                best_experiment = experiment
    return best_experiment


In [17]:
# These are example dcitionaries for each supported model type.

logistic_config = {
    'model_type': 'logistic_with_tfidf',
    'vectorizer': {'ngram_range': (1, 2), 'min_df': 10, 'stop_words': 'english'},
    'classifier': {'max_iter': 200},
    'expressions': {'expressions': [r'game|player', r'book|author|reader', r'song', r'dvd']}
}
multinomial_config = {
    'model_type': 'multinomial_count_vect',
    'vectorizer': {'ngram_range': (1, 2), 'min_df': 7, 'stop_words': 'english'},
    'classifier': {},
    # 'expressions': {'expressions': [r'game|player', r'book|author|reader', r'song', r'dvd']}
}
multinomial__with_features_config = {
    'model_type': 'multinomial_count_vect_with_features',
    'vectorizer': {'ngram_range': (1, 2), 'min_df': 7, 'stop_words': 'english'},
    'classifier': {},
    'expressions': {'expressions': [r'game|player', r'book|author|reader', r'song', r'dvd']}
}
model_config = {
    'model_type': 'voting',
    'logistic': logistic_config,
    'multinomial': multinomial_config
}

In [18]:
# Uncomment the next line to run a tiny experiment. Helpful for debugging.
# model = experiment(model_config, train[:1000], dev[:1000], 'data/test_experiment')


In [19]:
# Run the current model configuration experiment, log it and return it.
model = experiment(model_config, train, dev)


Experiment finished.
{'model_type': 'voting', 'logistic': {'model_type': 'logistic_with_tfidf', 'vectorizer': {'ngram_range': (1, 2), 'min_df': 10, 'stop_words': 'english'}, 'classifier': {'max_iter': 200}, 'expressions': {'expressions': ['game|player', 'book|author|reader', 'song', 'dvd']}}, 'multinomial': {'model_type': 'multinomial_count_vect', 'vectorizer': {'ngram_range': (1, 2), 'min_df': 7, 'stop_words': 'english'}, 'classifier': {}}}
{'macro_f1': 0.9140396829636389, 'confusion': [[7962, 77, 50], [166, 4190, 431], [67, 502, 4133]]}


## Error Analysis

In [20]:
def error_analysis(model, dev_data: pd.DataFrame, n_to_inspect=5):
    """Print simple error analysis with the dev data."""
    y_hat = model.predict(dev_data.text)
    y = dev_data.label
    results_df = dev_data.copy().assign(y_hat=y_hat)
    results_df[results_df.label != results_df.y_hat]

    errors = results_df

    # look at incorrect classificaitons
    mistake_types = [(i, j) for i, j in itertools.product(range(0, 3), repeat=2) if i != j]

    for true, pred in mistake_types:

        mistakes = errors[(errors.label == true) & (errors.y_hat == pred)]
        print(f"{LABELS[true]} which were classed as {LABELS[pred]}")
        n_iter = 0
        for _, mistake in mistakes.iterrows():
            print('\n'.join(textwrap.wrap('*  ' + mistake.text)))
            n_iter += 1
            if n_iter == n_to_inspect:
                break
        print()

error_analysis(model, dev)

non-movie which were classed as postive
*  A tale of several women forced to live through World War II in a
little village near London. A Jewish girl forced to marry a young man
she doesn't love in order to escape the Nazis. A impetuous, spoiled
young southern belle who marries a dashing young English Soldier to
escape the law.  A woman in this small village waiting for her dashing
young English Soldier to return to marry her only to find he's married
the southern belle and brought her back home. Along with others who
were all finally able to overcome biases, discrimination, loss, grief,
rationing, bombings and separation from loved ones to forge a bond
that survived everything in spite of it all. A moving tale of women
who became stronger than they every thought possible.
*  Best movie-based video game I've ever played. Like a mix between
Batman & Assassin's Creed
*  <div id="video-block-R3G9P1BL7ICV4Q" class="a-section a-spacing-
small a-spacing-top-mini video-block"></div><input typ

## Create Sumbission

In [21]:
test_data = load_data('data/test.csv')
test_data

,id,text
0,1087873697464833975,This tv series is one of the best I have seen ...
1,5853461517618045821,I saw Evanescence live this summer and it was ...
2,1225516091890726770,Almost from the word go this film is poor and ...
3,11795874926011119457,I bought this book and I know friends who have...
4,15956464546963841646,"“As you please.”\r\n\r\n“Ah, I forgot! A lette..."
...,...,...
17575,15791656241235236957,This was used to put on a mP3 player. Great so...
17576,3845684530705886088,"Contains spoilers<br /><br />""Hollow Man"" is p..."
17577,15849751097369757381,In relative terms having watched a lot of disg...
17578,17051107137572426951,"Paul W.S. Anderson is a crappy director, almos..."


In [ ]:
def ngram_to_tuple(config: dict):
    """Make sure that serialized config values are converted back to tuples."""
    for k, v in config.items():
        if isinstance(v, dict):
            config[k] = ngram_to_tuple(v)
        elif k == 'ngram_range':
            config[k] = tuple(v)
    return config

{'config': {'logistic': {'classifier': {'max_iter': 200},
                         'expressions': {'expressions': ['game|player',
                                                         'book|author|reader',
                                                         'song',
                                                         'dvd']},
                         'model_type': 'logistic_with_tfidf',
                         'vectorizer': {'min_df': 10,
                                        'ngram_range': (1, 2),
                                        'stop_words': 'english'}},
            'model_type': 'voting',
            'multinomial': {'classifier': {},
                            'model_type': 'multinomial_count_vect',
                            'vectorizer': {'min_df': 7,
                                           'ngram_range': (1, 2),
                                           'stop_words': 'english'}}},
 'confusion': [[7962, 77, 50], [166, 4190, 431], [67, 502, 4133]],
 'ma

In [34]:
# NOTE: this is shows some easy ways to load a model and train it.
# I was lazy and didn't actually save the models (worse case was like 5 minutes to train.)
best_config = ngram_to_tuple(load_best_config('data/experiments'))
pprint.pprint(best_config, sort_dicts=False)
model = make_model_pipeline(best_config['config'])
model

{'model_type': 'voting',
 'config': {'model_type': 'voting',
            'logistic': {'model_type': 'logistic_with_tfidf',
                         'vectorizer': {'ngram_range': (1, 2),
                                        'min_df': 10,
                                        'stop_words': 'english'},
                         'classifier': {'max_iter': 200},
                         'expressions': {'expressions': ['game|player',
                                                         'book|author|reader',
                                                         'song',
                                                         'dvd']}},
            'multinomial': {'model_type': 'multinomial_count_vect',
                            'vectorizer': {'ngram_range': (1, 2),
                                           'min_df': 7,
                                           'stop_words': 'english'},
                            'classifier': {}}},
 'macro_f1': 0.9140396829636389,
 'confusion':

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('multinomial', ...), ('logistic', ...)]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegression()
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",None
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict',

In [35]:
model.fit(df.text, df.label)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('multinomial', ...), ('logistic', ...)]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegression()
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",None
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict',

In [37]:
submission = test_data.assign(LABEL=model.predict(test_data.text)).rename(columns={'id': 'ID'})

In [39]:
submission.shape[0] == 17580

True

In [ ]:
sumbission[['ID', 'LABEL']].to_csv('submission.csv', index=False)